# 02 — Предобработка данных

**Цель:** построить единый воспроизводимый препроцессор для train и test без утечки целевой переменной.

На этом этапе модель не обучается. Мы только определяем правила преобразования признаков и проверяем, что train и test превращаются в совместимые матрицы.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from scipy import sparse

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features import (
    ABSENCE_CATEGORICAL_COLUMNS,
    CATEGORICAL_CODE_COLUMNS,
    build_preprocessor,
    get_feature_names,
    prepare_feature_frame,
    split_feature_types,
)

RAW_DIR = PROJECT_ROOT / "data" / "raw"
train = pd.read_csv(RAW_DIR / "train.csv")
test = pd.read_csv(RAW_DIR / "test.csv")

## 1. Отделяем идентификатор и целевую переменную

In [2]:
train_ids = train["Id"].copy()
test_ids = test["Id"].copy()
y = train["SalePrice"].copy()
y_log = np.log1p(y)

X_train_raw = train.drop(columns="SalePrice")
X_test_raw = test.copy()

print("X_train:", X_train_raw.shape)
print("X_test: ", X_test_raw.shape)
print("y:      ", y.shape)
print("Id сохранён отдельно:", train_ids.iloc[0], "...", test_ids.iloc[-1])

X_train: (1460, 80)
X_test:  (1459, 80)
y:       (1460,)
Id сохранён отдельно: 1 ... 2919


`SalePrice` не передаётся в препроцессор. `Id` хранится отдельно для будущего submission, но удаляется из матрицы признаков: его числовой порядок не описывает характеристики дома.

## 2. Исправляем смысл типов данных

In [3]:
prepared_preview = prepare_feature_frame(X_train_raw)
numeric_columns, categorical_columns = split_feature_types(prepared_preview)

type_summary = pd.DataFrame(
    {
        "Группа": ["Числовые", "Категориальные"],
        "Количество": [len(numeric_columns), len(categorical_columns)],
    }
)
display(type_summary)
print("Числовые коды, переведённые в категории:", CATEGORICAL_CODE_COLUMNS)
print("MSSubClass dtype:", prepared_preview["MSSubClass"].dtype)
print("MoSold dtype:", prepared_preview["MoSold"].dtype)

,Группа,Количество
0,Числовые,34
1,Категориальные,45


Числовые коды, переведённые в категории: ('MSSubClass', 'MoSold')
MSSubClass dtype: object
MoSold dtype: object


`MSSubClass` — код класса здания, а `MoSold` — номер месяца. Арифметическое расстояние между их значениями не означает сходство объектов, поэтому в базовом варианте мы рассматриваем их как категории. `YrSold` оставляем числовым, поскольку у года есть естественный временной порядок.

## 3. Разделяем виды пропусков

In [4]:
absence_columns = [
    column
    for column in ABSENCE_CATEGORICAL_COLUMNS
    if column in prepared_preview.columns
]
other_missing = [
    column
    for column in prepared_preview.columns[prepared_preview.isna().any()]
    if column not in absence_columns
]

missing_plan = pd.DataFrame(
    {
        "Группа": [
            "NA означает отсутствие объекта",
            "Числовые и обычные категориальные пропуски",
        ],
        "Правило": [
            "Отдельная категория None",
            "Медиана для числовых, мода для категориальных",
        ],
        "Количество столбцов": [len(absence_columns), len(other_missing)],
        "Столбцы": [", ".join(absence_columns), ", ".join(other_missing)],
    }
)
display(missing_plan)

,Группа,Правило,Количество столбцов,Столбцы
0,NA означает отсутствие объекта,Отдельная категория None,15,"Alley, MasVnrType, BsmtQual, BsmtCond, BsmtExp..."
1,Числовые и обычные категориальные пропуски,"Медиана для числовых, мода для категориальных",4,"LotFrontage, MasVnrArea, Electrical, GarageYrBlt"


Отсутствие бассейна, гаража или камина — содержательная характеристика дома. Поэтому такие `NA` не заменяются наиболее частым значением. Для числовых пропусков используются медианы и дополнительные индикаторы пропуска. Все значения вычисляются только во время `fit`.

## 4. Собираем и обучаем препроцессор

In [5]:
preprocessor = build_preprocessor(X_train_raw, scale_numeric=True)

X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)
feature_names = get_feature_names(preprocessor)

print("Матрица train:", X_train.shape)
print("Матрица test: ", X_test.shape)
print("Количество выходных признаков:", len(feature_names))
print("Формат разреженный:", sparse.issparse(X_train))

Матрица train:

 (1460, 330)
Матрица test:  (1459, 330)
Количество выходных признаков: 330
Формат разреженный: True


Препроцессор обучается через `fit_transform` только на train. Для test вызывается только `transform`, поэтому медианы, наиболее частые значения, масштабы и категории test не влияют на обучение.

## 5. Проверяем результат

In [6]:
def contains_nan(matrix):
    values = matrix.data if sparse.issparse(matrix) else np.asarray(matrix)
    return np.isnan(values).any()


checks = pd.Series(
    {
        "Число строк train сохранилось": X_train.shape[0] == len(train),
        "Число строк test сохранилось": X_test.shape[0] == len(test),
        "Число столбцов совпадает": X_train.shape[1] == X_test.shape[1],
        "В train нет NaN": not contains_nan(X_train),
        "В test нет NaN": not contains_nan(X_test),
        "Id исключён": not any(name.endswith("__Id") for name in feature_names),
        "SalePrice исключён": not any("SalePrice" in name for name in feature_names),
    },
    name="Проверка пройдена",
)
display(checks.to_frame())
assert checks.all()

,Проверка пройдена
Число строк train сохранилось,True
Число строк test сохранилось,True
Число столбцов совпадает,True
В train нет NaN,True
В test нет NaN,True
Id исключён,True
SalePrice исключён,True


In [7]:
neighborhood_features = [
    name for name in feature_names if "Neighborhood_" in name
]
print("One-Hot признаки Neighborhood:", len(neighborhood_features))
display(pd.Series(neighborhood_features, name="Выходные признаки Neighborhood").to_frame())

One-Hot признаки Neighborhood: 25


,Выходные признаки Neighborhood
0,categorical__Neighborhood_Blmngtn
1,categorical__Neighborhood_Blueste
2,categorical__Neighborhood_BrDale
3,categorical__Neighborhood_BrkSide
4,categorical__Neighborhood_ClearCr
5,categorical__Neighborhood_CollgCr
6,categorical__Neighborhood_Crawfor
7,categorical__Neighborhood_Edwards
8,categorical__Neighborhood_Gilbert
9,categorical__Neighborhood_IDOTRR


`Neighborhood` не получает искусственного числового порядка. Каждый район представлен отдельным One-Hot-признаком. Кодировщик настроен с `handle_unknown="ignore"`, поэтому новая категория в будущем наборе данных не приведёт к ошибке.

## 6. Почему pipeline защищает от leakage

1. `SalePrice` отделяется до преобразований.
2. `Id` не используется как признак.
3. Медианы, моды, масштабирование и список категорий вычисляются при `fit` только по обучающим строкам.
4. В будущем препроцессор будет частью общего `Pipeline` с моделью. При кросс-валидации он будет заново обучаться внутри каждого train-fold.
5. Test используется только через `transform`.

Сохранять преобразованные train и test как главный источник данных не нужно. Воспроизводимый pipeline должен каждый раз начинать с неизменённых CSV из `data/raw`.

## 7. Результат этапа

Получен единый препроцессор, который:

- корректирует типы `MSSubClass` и `MoSold`;
- различает смысловые и обычные пропуски;
- заполняет числовые значения медианой и создаёт индикаторы пропусков;
- кодирует категории One-Hot, включая `Neighborhood`;
- масштабирует числовые признаки;
- выдаёт train и test с одинаковым числом столбцов и без `NaN`;
- не использует `SalePrice` или `Id` в признаках.

Следующий этап — подключить этот препроцессор к нескольким регрессионным моделям и сравнить их по кросс-валидации. На данном этапе модели не обучались.